In [ ]:

import openai
from web_tool import WebSearchTool, tavily_tool
from rag_tool import theory_engine, stats_engine


class TextualAgent:
    def __init__(self, rag_tool):
        self.rag_tool = rag_tool
        self.system_prompt = (
            "Tu es un expert en analyse textuelle. Explique les concepts pédagogiquement. "
            "CONSIGNE : Ne donne JAMAIS de formules, concentre-toi sur les idées."
            "RÈGLE CRUCIALE : A la fin de chaque paragraphe ou explication, cite la source utilisée entre parenthèses, par exemple : (Trouvé sur: Cours_ML_Stats_Chap2.pdf"
        )

    def answer(self, user_query: str):
        # CHANGEMENT : On appelle la méthode spécifique "theory_engine" pour le moteur de recherche de théorie
        context = self.rag_tool.run(user_query) 
        
        messages = [
            {"role": "system", "content": self.system_prompt},
            {"role": "user", "content": f"CONTEXTE :\n{context}\n\nQUESTION : {user_query}"}
        ]
        response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages, temperature=0.2)
        return response.choices[0].message.content

class MathAgent:
    def __init__(self, rag_tool):
        self.rag_tool = rag_tool
        self.system_prompt = (
            "Tu es un expert en mathématiques. Extrait les formules en format LaTeX ($$)."
            "Si aucune formule n'est trouvée, dis : 'Aucune équation détectée'."
            "RÈGLE CRUCIALE : A la fin de chaque paragraphe ou explication, cite la source utilisée entre parenthèses, par exemple : (Trouvé sur: Cours_ML_Stats_Chap2.pdf"
            "Si vraiment aucun symbole mathématique n'existe pas dans le texte, alors dis : 'Aucune équation détectée'."
        )

    def answer(self, user_query: str):
        # CHANGEMENT : On appelle la méthode spécifique "stats_engine" pour le moteur de recherche de statistiques
        context = self.rag_tool.run(user_query)

        messages = [
            {"role": "system", "content": self.system_prompt},
            {"role": "user", "content": f"CONTEXTE :\n{context}\n\nQUESTION : {user_query}"}
        ]
        response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages, temperature=0.2)
        return response.choices[0].message.content



class WebAgent:
    def __init__(self):
        self.tool = tavily_tool  # On utilise l'outil que tu as créé
        self.system_prompt = (
            "Tu es un expert en recherche d'informations sur le web. "
            "Ta mission est de synthétiser les résultats trouvés de manière claire. "
            "RÈGLE : Cite toujours tes sources avec les liens URL fournis par l'outil."
        )

    def answer(self, user_query: str):
        # 1. L'agent utilise l'outil pour obtenir les données brutes
        raw_results = self.tool.search(user_query)
        
        # 2. L'agent envoie ces données au LLM pour rédaction
        messages = [
            {"role": "system", "content": self.system_prompt},
            {"role": "user", "content": f"RÉSULTATS DE RECHERCHE :\n{raw_results}\n\nQUESTION : {user_query}"}
        ]
        
        response = openai.chat.completions.create(
            model="gpt-4o-mini", 
            messages=messages, 
            temperature=0.3
        )
        return response.choices[0].message.content
    
# --- BLOC DE TEST POUR TESTER LES 3 AGENTS ---

if __name__ == "__main__":
    import os
    from dotenv import load_dotenv

    load_dotenv()

    # 1. Initialisation des agents
    # On passe les instances déjà créées dans rag_tool.py
    print("🤖 Initialisation des agents...")
    agent_texte = TextualAgent(rag_tool=theory_engine)
    agent_maths = MathAgent(rag_tool=stats_engine)
    agent_web = WebAgent()

    print("✅ Système prêt. Tapez 'exit' pour quitter.")

    while True:
        print("\n" + "="*50)
        print("CHOISISSEZ UN AGENT :")
        print("1. Agent Textuel (Théorie locale)")
        print("2. Agent Math (Formules locales)")
        print("3. Agent Web (Recherche Internet)")
        
        choix = input("\nVotre choix (1, 2, 3) : ").strip()

        if choix.lower() in ['exit', 'q', 'quit']:
            break

        query = input("Votre question : ").strip()
        if not query:
            continue

        print("\n🧠 Réflexion de l'agent...")

        try:
            if choix == "1":
                # L'agent texte appelle theory_engine.run() en interne
                reponse = agent_texte.answer(query)
                print(f"\n📝 [RÉPONSE THÉORIQUE] :\n{reponse}")
            
            elif choix == "2":
                # L'agent math appelle stats_engine.run() en interne
                reponse = agent_maths.answer(query)
                print(f"\n🔢 [RÉPONSE MATH] :\n{reponse}")
            
            elif choix == "3":
                # L'agent web appelle tavily_tool.search() en interne
                reponse = agent_web.answer(query)
                print(f"\n🌐 [RÉPONSE WEB] :\n{reponse}")
            
            else:
                print("⚠️ Choix invalide, veuillez taper 1, 2 ou 3.")

        except Exception as e:
            print(f"❌ Une erreur est survenue : {e}")